# Predictive Circuit Coding Training

This notebook is the Colab-first training surface. It installs the repo first, then uses package helpers for stage banners, preflight checks, and checkpoint messaging.

To train on only a subset of the canonical processed dataset, edit the `dataset_selection` block in `configs/pcc/predictive_circuit_coding_base.yaml` before launching `pcc-train`.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
repo_root = Path('/content/predictive-circuit-coding')
drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = drive_root / 'data' / 'allen_visual_behavior_neuropixels'
drive_artifacts_root = drive_root / 'artifacts'
drive_root.mkdir(parents=True, exist_ok=True)
drive_artifacts_root.mkdir(parents=True, exist_ok=True)
print('Drive mounted. Repo root:', repo_root)

In [ ]:
if not repo_root.exists():
    !git clone https://github.com/jacobposchl/predictive-circuit-coding.git {repo_root}
%cd {repo_root}
!pip install -e .[notebook]

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if repo_artifacts_root.exists() or repo_artifacts_root.is_symlink():
    if repo_artifacts_root.is_symlink():
        repo_artifacts_root.unlink()
    else:
        shutil.rmtree(repo_artifacts_root)
repo_artifacts_root.symlink_to(drive_artifacts_root, target_is_directory=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Linked artifacts into repo:', repo_artifacts_root)

In [ ]:
from predictive_circuit_coding.utils import NotebookStageReporter, verify_paths_exist

reporter = NotebookStageReporter(name='colab-train', expected_duration='2-10 minutes for debug runs, longer for full A100 runs')
reporter.banner('Predictive Circuit Coding Training', subtitle='Setup, preflight, training, and evaluation')
experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
resume_checkpoint = checkpoint_dir / 'pcc_best.pt'

In [ ]:
reporter.begin('preflight', next_artifact='best checkpoint + evaluation summary')
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing config files for training notebook.'
print('Checkpoint resume path:', resume_checkpoint)
print('Dataset selection is controlled by the dataset_selection block in the experiment config.')
print('If you want a reusable derived subset manifest before training, run pcc-prepare-data materialize-runtime-selection with the same configs.')
print('If you want to resume, point training.resume_checkpoint in the experiment config at an existing checkpoint.')
reporter.finish('preflight')

In [ ]:
reporter.begin('training', next_artifact='artifacts/checkpoints/pcc_best.pt')
%cd {repo_root}
!pcc-train --config {experiment_config} --data-config {data_config}
reporter.note_checkpoint(resume_checkpoint)
reporter.finish('training')

In [ ]:
reporter.begin('evaluation', next_artifact='validation-ready evaluation summary json')
%cd {repo_root}
!pcc-evaluate --config {experiment_config} --data-config {data_config} --checkpoint {resume_checkpoint} --split valid
reporter.finish('evaluation')

In [ ]:
from IPython.display import Javascript, display

display(Javascript('google.colab.kernel.disconnect()'))